### Домашнее задание: переходим к нейросетям (6 баллов)

В предыдущем ноутбуке мы изучили статистические языковые модели. Теперь посмотрим, что предлагает глубокое обучение.

<img src='https://raw.githubusercontent.com/yandexdataschool/nlp_course/master/resources/expanding_mind_lm_kn_3.png' width=300px>

Будем работать с тем же датасетом, но построим **символьную**, а не словную языковую модель.

Перед началом: если ты ещё не проходил семинар, открой `seminar.ipynb` и скачай данные.

> Неофициальный русский перевод [оригинального задания](https://github.com/yandexdataschool/nlp_course/blob/2026/week02_lm/homework_pytorch.ipynb). Вычисления и проверки сохранены.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

При работе на уровне символов нам не нужен огромный словарь, и проблема неизвестных слов исчезает. Более того, можно сохранять прописные буквы! Однако длина последовательностей существенно увеличится.

Нам по-прежнему нужны специальные токены:

- **Begin Of Sequence (BOS)** — начало последовательности. Ставится перед каждым текстом, чтобы вход нейросети никогда не был пустым. $P(x_t) = P(x_1 | BOS)$.
- **End Of Sequence (EOS)** — конец последовательности. Встречается **только в самом конце**. Если модель сгенерировала EOS, генерация завершена.

In [ ]:
BOS, EOS = ' ', '\n'

data = pd.read_json("./arxivData.json")
lines = data.apply(lambda row: (row['title'] + ' ; ' + row['summary'])[:512], axis=1) \
            .apply(lambda line: BOS + line.replace(EOS, ' ') + EOS) \
            .tolist()

# если пропустил семинар, скачай данные здесь: https://yadi.sk/d/_nGyU2IajjR9-w

Следующий шаг — **построить словарь символов**: собери список всех уникальных символов датасета.

In [ ]:
# получи все уникальные символы из строк, включая заглавные буквы и знаки
tokens = <YOUR CODE>

tokens = sorted(tokens)
n_tokens = len(tokens)
print ('n_tokens = ',n_tokens)
assert 100 < n_tokens < 150
assert BOS in tokens, EOS in tokens

Теперь каждому символу можно присвоить индекс в списке токенов, чтобы кодировать строки целочисленными векторами, пригодными для PyTorch.

In [ ]:
# словарь: символ → идентификатор (индекс в списке tokens)
token_to_id = <YOUR CODE>

In [ ]:
assert len(tokens) == len(token_to_id), "dictionaries must have same size"
for i in range(n_tokens):
    assert token_to_id[tokens[i]] == i, "token identifier must be it's position in tokens list"

print("Seems alright!")

Осталось объединить несколько строк в целочисленную матрицу формы `[batch_size, text_length]`.

Проблема: у строк разные длины. Её можно решить, дополняя короткие строки токенами `EOS` или обрезая длинные. Ниже показано как.

In [ ]:
def to_matrix(lines, max_len=None, pad=token_to_id[EOS], dtype=np.int64):
    """Casts a list of lines into torch-digestable matrix"""
    max_len = max_len or max(map(len, lines))
    lines_ix = np.full([len(lines), max_len], pad, dtype=dtype)
    for i in range(len(lines)):
        line_ix = list(map(token_to_id.get, lines[i][:max_len]))
        lines_ix[i, :len(line_ix)] = line_ix
    return lines_ix

In [ ]:
# Пример: объедини четыре случайных имени в одну матрицу, добавляя нули для padding.
dummy_lines = [
    ' abc\n',
    ' abacaba\n',
    ' abc1234567890\n',
]
print(to_matrix(dummy_lines))



### Нейросетевая языковая модель (2 балла, включая обучение)

Как и для N-граммных моделей, нам нужно оценить вероятность текста как совместную вероятность его токенов (теперь — символов):

$$P(X) = \prod_t P(x_t \mid x_0, \dots, x_{t-1}).$$

Но вместо подсчёта всех статистик обучим нейросеть с параметрами $\theta$, приближающую условные вероятности:

$$ P(x_t \mid x_0, \dots, x_{t-1}) \approx p(x_t \mid x_0, \dots, x_{t-1}, \theta) $$

Прежде чем оптимизировать параметры, нужно определить архитектуру сети. Начнём с архитектуры с фиксированным окном (**свёрточной**):

<img src='https://raw.githubusercontent.com/yandexdataschool/nlp_course/master/resources/fixed_window_lm.jpg' width=400px>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class FixedWindowLanguageModel(nn.Module):
    def __init__(self, n_tokens=n_tokens, emb_size=16, hid_size=64):
        """
        A fixed window model that looks on at least 5 previous symbols.

        Note: fixed window LM is effectively performing a convolution over a sequence of words.
        This convolution only looks on current and previous words.
        Such convolution can be represented as a sequence of 2 operations:
        - pad input vectors by {filter_size - 1} zero vectors on the "left", do not pad right
        - perform regular convolution with {filter_size} and stride 1

        - If you're absolutely lost, here's a hint: use nn.ZeroPad2d((NUM_LEADING_ZEROS, 0, 0, 0))
          followed by a nn.Conv1d(..., padding=0). And yes, its okay that padding is technically "2d".

        Feel free to try different window and model sizes to find configuration you'll like.
        """
        super().__init__() # initialize base class to track sub-layers, trainable variables, etc.

        # ТВОЙ КОД - create layers/variables and any metadata you want, e.g. self.emb = L.Embedding(...)

        <...>

        # КОНЕЦ ТВОЕГО КОДА

    def __call__(self, input_ix):
        """
        compute language model logits given input tokens
        :param input_ix: batch of sequences with token indices, tensor: int32[batch_size, sequence_length]
        :returns: pre-softmax linear outputs of language model [batch_size, sequence_length, n_tokens]
            these outputs will be used as logits to compute P(x_t | x_0, ..., x_{t - 1})

        :note: that convolutions operate with tensors of shape [batch, channels, length], while linear layers
         and *embeddings* use [batch, length, channels] tensors. Use tensor.permute(...) to adjust shapes.

        """
        # ТВОЙ КОД - apply layers, see docstring above
        return <...> # output tensor should be of shape [batch_size, sequence_length, n_tokens]

    def get_possible_next_tokens(self, prefix=BOS, temperature=1.0, max_len=100):
        """ :returns: probabilities of next token, dict {token : prob} for all tokens """
        prefix_ix = torch.as_tensor(to_matrix([prefix]), dtype=torch.int64)
        with torch.no_grad():
            probs = torch.softmax(self(prefix_ix)[0, -1], dim=-1).cpu().numpy()  # shape: [n_tokens]
        return dict(zip(tokens, probs))


In [ ]:
dummy_model = FixedWindowLanguageModel()

dummy_input_ix = torch.as_tensor(to_matrix(dummy_lines))
dummy_logits = dummy_model(dummy_input_ix)

print('Weights:', tuple(name for name, w in dummy_model.named_parameters()))

In [ ]:
assert isinstance(dummy_logits, torch.Tensor)
assert dummy_logits.shape == (len(dummy_lines), max(map(len, dummy_lines)), n_tokens), "please check output shape"
assert np.all(np.isfinite(dummy_logits.data.cpu().numpy())), "inf/nan encountered"
assert not np.allclose(dummy_logits.data.cpu().numpy().sum(-1), 1), "please predict linear outputs, don't use softmax (maybe you've just got unlucky)"

In [ ]:
# проверка отсутствия заглядывания в будущие токены
dummy_input_ix_2 = torch.as_tensor(to_matrix([line[:3] + 'e' * (len(line) - 3) for line in dummy_lines]))
dummy_logits_2 = dummy_model(dummy_input_ix_2)

assert torch.allclose(dummy_logits[:, :3], dummy_logits_2[:, :3]), "your model's predictions depend on FUTURE tokens. " \
    " Make sure you don't allow any layers to look ahead of current token." \
    " You can also get this error if your model is not deterministic (e.g. dropout). Disable it for this test."

Теперь можно подбирать параметры сети, минимизируя категориальную кросс-энтропию на обучающем датасете $D$:

$$ L = {\frac1{|D|}} \sum_{X \in D} \sum_{x_i \in X} - \log p(x_t \mid x_1, \dots, x_{t-1}, \theta) $$

Как обычно, оптимизация нейросети выполняется стохастическим градиентным спуском с обратным распространением ошибки.

Минимизация кросс-энтропии связана с минимизацией **перплексии** и KL-дивергенции и максимизацией логарифма правдоподобия.

In [ ]:
def compute_mask(input_ix, eos_ix=token_to_id[EOS]):
    """ compute a boolean mask that equals "1" until first EOS (including that EOS) """
    return F.pad(torch.cumsum(input_ix == eos_ix, dim=-1)[..., :-1] < 1, pad=(1, 0, 0, 0), value=True)

print('matrix:\n', dummy_input_ix.numpy())
print('mask:', compute_mask(dummy_input_ix).to(torch.int32).cpu().numpy())
print('lengths:', compute_mask(dummy_input_ix).sum(-1).cpu().numpy())

In [ ]:
def compute_loss(model, input_ix):
    """
    :param model: language model that can compute next token logits given token indices
    :param input ix: int32 matrix of tokens, shape: [batch_size, length]; padded with eos_ix
    :returns: scalar loss function, mean crossentropy over non-eos tokens
    """
    input_ix = torch.as_tensor(input_ix, dtype=torch.int64)


    logits = model(input_ix[:, :-1])
    reference_answers = input_ix[:, 1:]

    # Задание: реализуй loss по формуле выше
    # вычисляй loss только по настоящим токенам, исключая padding
    # настоящие токены и первый EOS учитываются, последующие EOS — нет
    # при желании используй функцию compute_mask из предыдущей ячейки

    <YOUR CODE>

    return <YOUR CODE: return scalar loss>



In [ ]:
loss_1 = compute_loss(dummy_model, to_matrix(dummy_lines, max_len=15))
loss_2 = compute_loss(dummy_model, to_matrix(dummy_lines, max_len=16))
assert (np.ndim(loss_1) == 0) and (0 < loss_1 < 100), "loss must be a positive scalar"
assert torch.allclose(loss_1, loss_2), 'do not include padding EOS tokens AFTER the first EOS into loss. '\
    'Hint: use compute_mask. Beware +/-1 errors. And be careful when averaging!'

### Оценка качества

Тебе понадобятся две функции: одна для вычисления ошибки на тестовой выборке, вторая для генерации примеров. Для удобства мы уже реализовали обе.

In [ ]:
def score_lines(model, dev_lines, batch_size):
    """ computes average loss over the entire dataset """
    dev_loss_num, dev_loss_len = 0., 0.
    with torch.no_grad():
        for i in range(0, len(dev_lines), batch_size):
            batch_ix = to_matrix(dev_lines[i: i + batch_size])
            dev_loss_num += compute_loss(model, batch_ix).item() * len(batch_ix)
            dev_loss_len += len(batch_ix)
    return dev_loss_num / dev_loss_len

def generate(model, prefix=BOS, temperature=1.0, max_len=100):
    """
    Samples output sequence from probability distribution obtained by model
    :param temperature: samples proportionally to model probabilities ^ temperature
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    with torch.no_grad():
        while True:
            token_probs = model.get_possible_next_tokens(prefix)
            tokens, probs = zip(*token_probs.items())
            if temperature == 0:
                next_token = tokens[np.argmax(probs)]
            else:
                probs = np.array([p ** (1. / temperature) for p in probs])
                probs /= sum(probs)
                next_token = np.random.choice(tokens, p=probs)

            prefix += next_token
            if next_token == EOS or len(prefix) > max_len: break
    return prefix

### Цикл обучения

Наконец обучим модель на мини-батчах данных.

In [ ]:
from sklearn.model_selection import train_test_split
train_lines, dev_lines = train_test_split(lines, test_size=0.25, random_state=42)

batch_size = 256
score_dev_every = 250
train_history, dev_history = [], []
model = FixedWindowLanguageModel()
opt = torch.optim.Adam(model.parameters())

# подсказка: если хочешь использовать CUDA, сейчас самое время

# оцени необученную модель
dev_history.append((0, score_lines(model, dev_lines, batch_size)))
print("Sample before training:", generate(model, 'Bridging'))

In [ ]:
from IPython.display import clear_output
from random import sample
from tqdm import trange

for i in trange(len(train_history), 5000):
    batch = to_matrix(sample(train_lines, batch_size))


    loss_i = compute_loss(model, batch)

    opt.zero_grad()
    loss_i.backward()
    opt.step()

    train_history.append((i, loss_i.item()))

    if (i + 1) % 50 == 0:
        clear_output(True)
        plt.scatter(*zip(*train_history), alpha=0.1, label='train_loss')
        if len(dev_history):
            plt.plot(*zip(*dev_history), color='red', label='dev_loss')
        plt.legend(); plt.grid(); plt.show()
        print("Generated examples (tau=0.5):")
        for _ in range(3):
            print(generate(model, temperature=0.5))

    if (i + 1) % score_dev_every == 0:
        print("Scoring dev...")
        dev_history.append((i, score_lines(model, dev_lines, batch_size)))
        print('#%i Dev loss: %.3f' % dev_history[-1])


In [ ]:
assert np.mean(train_history[:10], axis=0)[1] > np.mean(train_history[-10:], axis=0)[1], "The model didn't converge."
print("Final dev loss:", dev_history[-1][-1])

for i in range(10):
    print(generate(model, temperature=0.5))

**Какими будут сгенерированные тексты?** Модель посимвольная, поэтому скорее всего получится похожая на научные тексты смесь символов: фрагменты слов, характерная для arXiv пунктуация и очень мало настоящего смысла. Это ожидаемый результат, а не признак ошибки. Оценивай модель по прохождению проверки сходимости и по ошибке на валидационной выборке, а не по читаемости примеров.

Возможно, RNN из следующего раздела при настройках по умолчанию покажется заметно лучше. Но сравнение не вполне честное: здесь `hid_size=64`, а для RNN — `hid_size=256`. Если разница тебя удивляет, сначала попробуй сопоставимую скрытую размерность для модели с фиксированным окном (а возможно, и вторую свёртку), прежде чем делать вывод, что дело в самой архитектуре.

### Языковые модели на основе RNN (3 балла, включая обучение)

Архитектуры с фиксированным окном хорошо захватывают краткосрочные зависимости, но не могут учитывать информацию вне этого окна.

Проблему можно ослабить с помощью **рекуррентной нейросети (RNN)**:

$$ h_0 = \vec 0 ; \quad h_{t+1} = RNN(x_t, h_t) $$

$$ p(x_t \mid x_0, \dots, x_{t-1}, \theta) = dense_{softmax}(h_{t-1}) $$

Такая модель читает текст по одному токену слева направо и переносит между шагами вектор скрытого состояния. Теоретически, при достаточной размерности этого вектора она способна изучить зависимости произвольной длины.

<img src='https://raw.githubusercontent.com/yandexdataschool/nlp_course/master/resources/rnn_lm.jpg' width=480px>

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, n_tokens=n_tokens, emb_size=16, hid_size=256):
        """
        Build a recurrent language model.
        You are free to choose anything you want, but the recommended architecture is
        - token embeddings
        - one or more LSTM/GRU layers with hid size
        - linear layer to predict logits

        :note: if you use nn.RNN/GRU/LSTM, make sure you specify batch_first=True
         With batch_first, your model operates with tensors of shape [batch_size, sequence_length, num_units]
         Also, please read the docs carefully: they don't just return what you want them to return :)
        """
        super().__init__() # initialize base class to track sub-layers, trainable variables, etc.

        # ТВОЙ КОД - create layers/variables/etc

        <...>

        # КОНЕЦ ТВОЕГО КОДА

    def __call__(self, input_ix):
        """
        compute language model logits given input tokens
        :param input_ix: batch of sequences with token indices, tensor: int32[batch_size, sequence_length]
        :returns: pre-softmax linear outputs of language model [batch_size, sequence_length, n_tokens]
            these outputs will be used as logits to compute P(x_t | x_0, ..., x_{t - 1})
        """
        # ТВОЙ КОД - apply layers, see docstring above
        return <...> # output tensor should be of shape [batch_size, sequence_length, n_tokens]

    def get_possible_next_tokens(self, prefix=BOS, temperature=1.0, max_len=100):
        """ :returns: probabilities of next token, dict {token : prob} for all tokens """
        prefix_ix = torch.as_tensor(to_matrix([prefix]), dtype=torch.int64)
        with torch.no_grad():
            probs = torch.softmax(self(prefix_ix)[0, -1], dim=-1).cpu().numpy()  # shape: [n_tokens]
        return dict(zip(tokens, probs))


In [ ]:
model = RNNLanguageModel()

dummy_input_ix = torch.as_tensor(to_matrix(dummy_lines))
dummy_logits = model(dummy_input_ix)

assert isinstance(dummy_logits, torch.Tensor)
assert dummy_logits.shape == (len(dummy_lines), max(map(len, dummy_lines)), n_tokens), "please check output shape"
assert not np.allclose(dummy_logits.cpu().data.numpy().sum(-1), 1), "please predict linear outputs, don't use softmax (maybe you've just got unlucky)"
print('Weights:', tuple(name for name, w in model.named_parameters()))

In [ ]:
# проверка отсутствия заглядывания в будущие токены
dummy_input_ix_2 = torch.as_tensor(to_matrix([line[:3] + 'e' * (len(line) - 3) for line in dummy_lines]))
dummy_logits_2 = model(dummy_input_ix_2)

assert torch.allclose(dummy_logits[:, :3], dummy_logits_2[:, :3]), "your model's predictions depend on FUTURE tokens. " \
    " Make sure you don't allow any layers to look ahead of current token." \
    " You can also get this error if your model is not deterministic (e.g. dropout). Disable it for this test."

### Обучение RNN

Наша RNN должна минимизировать ту же функцию потерь, что и модель с фиксированным окном. Однако есть проблема: RNN многократно перемножает градиенты при распространении по временным шагам, и они могут **взрываться**, [нарушая обучение](https://raw.githubusercontent.com/yandexdataschool/nlp_course/master/resources/nan.jpg).

Обычно это решается **обрезкой градиентов (gradient clipping)** — [по значению](https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_value_.html) или [по общей норме](https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html).

**Задание:** реализуй код обучения с минимизацией функции потерь. Если наблюдаешь сильные колебания loss, добавь clipping: вызывай его после `loss_i.backward()`, но **до** `opt.step()`. Если проблем с градиентами нет, clipping не обязателен.

_Примечание: gradient clipping применим не только к RNN. Глубокие свёрточные сети тоже могут сталкиваться с такой проблемой._

In [ ]:
batch_size = 64         # <-- please tune batch size to fit your CPU/GPU configuration
score_dev_every = 250
train_history, dev_history = [], []

model = <YOUR CODE HERE>
opt = torch.optim.Adam(model.parameters())

# оцени необученную модель
dev_history.append((0, score_lines(model, dev_lines, batch_size)))
print("Sample before training:", generate(model, 'Bridging'))

In [ ]:
from IPython.display import clear_output
from random import sample
from tqdm import trange

for i in trange(len(train_history), 5000):
    batch = to_matrix(sample(train_lines, batch_size))

    <YOUR CODE - one step of the training loop for your RNN model>

    loss_i = <...>

    train_history.append((i, float(loss_i)))

    if (i + 1) % 50 == 0:
        clear_output(True)
        plt.scatter(*zip(*train_history), alpha=0.1, label='train_loss')
        if len(dev_history):
            plt.plot(*zip(*dev_history), color='red', label='dev_loss')
        plt.legend(); plt.grid(); plt.show()
        print("Generated examples (tau=0.5):")
        for _ in range(3):
            print(generate(model, temperature=0.5))

    if (i + 1) % score_dev_every == 0:
        print("Scoring dev...")
        dev_history.append((i, score_lines(model, dev_lines, batch_size)))
        print('#%i Dev loss: %.3f' % dev_history[-1])


In [ ]:
assert np.mean(train_history[:10], axis=0)[1] > np.mean(train_history[-10:], axis=0)[1], "The model didn't converge."
print("Final dev loss:", dev_history[-1][-1])
for i in range(10):
    print(generate(model, temperature=0.5))

### Альтернативные стратегии генерации (1 балл)

Пока мы выбирали токены пропорционально их вероятностям. Но даже бессмысленные последовательности никогда не имеют строго нулевой вероятности после softmax — поэтому иногда генерируются странные слова.

Уменьшение температуры частично помогает, но снижает разнообразие. Можно ли избавиться от маловероятного мусора, сохранив разнообразие? **Да, изменив стратегию выборки.**

**Top-k sampling:** на каждом шаге выбирать следующий токен только из **k самых вероятных**.

Пусть $k=3$, а вероятности токенов равны $p=[0.1, 0.35, 0.05, 0.2, 0.3]$. Выбираем три наиболее вероятных и обнуляем остальные:

$\hat p=[0.0, 0.35, 0.0, 0.2, 0.3]$

После нормализации получаем:

$p^*\approx[0.0, 0.412, 0.0, 0.235, 0.353]$.

**Nucleus sampling (top-p):** похожий метод, но количество подходящих токенов выбирается динамически. Мы включаем токены, суммарная вероятность которых покрывает заданную долю массы распределения.

Для $p=[0.1, 0.35, 0.05, 0.2, 0.3]$ и порога $N=0.9$:

1. Берём наиболее вероятный токен $w_2$, так как $p(w_2)<N$.
2. Добавляем $w_5$: $p(w_2)+p(w_5)=0.65<N$.
3. Добавляем $w_4$: $p(w_2)+p(w_5)+p(w_4)=0.85<N$.
4. Следующий кандидат уже превысит порог: $p(w_2)+p(w_5)+p(w_4)+p(w_1)=0.95>N$. На этом останавливаемся.

После отбора нормализуй оставшиеся вероятности, как в top-k, и генерируй следующий токен.

**Твоя задача:** реализовать вариант nucleus sampling и проверить качество генерации.

In [ ]:
def generate_nucleus(model, prefix=BOS, nucleus=0.9, max_len=100):
    r"""
    Generate a sequence with nucleus sampling
    :param prefix: a string of characters generated so far, which the model continues
    :param nucleus: N from the formulae above, N \in [0, 1]
    :param max_len: generate sequences with at most this many tokens, including prefix

    :note: make sure that nucleus always contains at least one word, even if p(w*) > nucleus

    """
    while True:
        token_probs = model.get_possible_next_tokens(prefix)
        tokens, probs = zip(*token_probs.items())

        <YOUR CODE HERE: restrict tokens to the nucleus, renormalize, then sample>

        next_token = <YOUR CODE>
        prefix += next_token
        if next_token == EOS or len(prefix) > max_len: break
    return prefix

In [ ]:
for i in range(10):
    print(generate_nucleus(model, nucleus=PLAY_WITH_ME_SENPAI))

### Дополнительная задача I: Beam Search (2 балла, включая примеры)

Иногда не нужно разнообразие: требуется **одна наиболее вероятная гипотеза** — например, лучший перевод или наиболее вероятное продолжение поискового запроса.

Точно найти такую последовательность сложно: для десяти токенов потребовалось бы перебрать все `|V|^10` комбинаций. На практике обычно используют приближённый алгоритм **beam search (лучевой поиск)**.

Алгоритм:

0. Инициализировать `beam = [prefix]`, максимальный размер луча `beam_size = k`.
1. Для каждого из `T` шагов:
2. Сгенерировать все возможные следующие токены для каждой гипотезы: `len(beam) * len(vocab)` кандидатов.
3. Оставить `beam_size` лучших кандидатов как новый `beam`.
4. После завершения выбрать лучшую гипотезу (или несколько) из `beam`.

In [ ]:
from IPython.display import HTML
# пример результата:
!wget -q https://raw.githubusercontent.com/yandexdataschool/nlp_course/2020/resources/beam_search.html
HTML("beam_search.html")

In [ ]:
def generate_beamsearch(model, prefix=BOS, beam_size=4, length=5):
    """
    Generate a sequence with beam search
    :param prefix: a string containing space-separated previous tokens
    :param beam_size: maintain this many top-k hypotheses in beam
    :param length: generate sequences with at most this many tokens, NOT INCLUDING PREFIX
    :returns: beam_size most likely candidates
    """

    <YOUR CODE HERE>

    return <most likely sequence>

In [ ]:
generate_beamsearch(model, prefix=' deep ', beam_size=4)

In [ ]:
# проверь, какой размер луча работает лучше
# найди как минимум пять префиксов, для которых beam_size=1 и beam_size=8 дают разные последовательности

### Дополнительная задача II: предельная языковая модель (2+ балла)

Ты уже знаешь основные строительные блоки нейросетевых языковых моделей. Попробуй собрать собственную максимально мощную модель:

- Выбери уровень токенизации: символы, слова или подслова, например [BPE](https://github.com/rsennrich/subword-nmt).
- Комбинируй свёртки, рекуррентные ячейки, предобученные эмбеддинги и другие методы глубокого обучения.
  - Используй stride для быстрого расширения рецептивного поля. [Схема из Google WaveNet](https://storage.googleapis.com/deepmind-live-cms/documents/BlogPost-Fig2-Anim-160908-r01.gif).
- Обучайся на большом объёме данных, например [1 Billion Words](http://www.statmt.org/lm-benchmark/1-billion-word-language-modeling-benchmark-r13output.tar.gz).
- Настраивай расписание обучения: сначала короткие последовательности, затем более длинные; см. стратегию [one-cycle learning rate](https://medium.com/@nachiket.tanksale/finding-good-learning-rate-and-the-one-cycle-policy-7159fe1db5d6).

**Сдавать эту задачу необязательно.** Не трать на неё столько времени, чтобы пропустить дедлайн остальных заданий.